In [1]:
import serial
import time
from datetime import datetime

from analysis_lib import KeysightInfiniiVisionMSO, DEFAULT_LABELS
from analysis_lib.decoder import RawDecoder, RawDecodeResult, RawFrame, BrotherSerialDecoder, BrotherDecodeResult, DecodedByte

ADDRESS = "USB0::2391::5925::MY49110266::INSTR"

INTERFACE = "IF60"
TYPEWRITER = "AX20"
KEYBOARD = 1 # Typenrad Selection Switch: 1=LOCAL, 2=INTERNATIONAL, 3=SYMBOL

IFPITCH = 15
switches = [
    0,  # 1-1: UP=RS-232C, DOWN=CDCC interface
    0,  # 1-2: UP=Terminal mode, DOWN=Printer mode
    0,  # 1-3: USA users can ignore
    1,  # 1-4: UP=ASCII Wheel, DOWN=Non-ASCII Wheel
    1,  # 1-5: UP=12-inch paper, DOWN=11-inch paper
    1,  # 1-6: UP=Auto skip perforation, DOWN=Non auto skip
        #
    1,  # 2-1: UP=Local echo (half-duplex), DOWN=No echo (full-duplex)
    1,  # 2-2: UP=DC-1/DC-3 disabled, DOWN=enabled
    1,  # 2-3: UP=Auto line feed off, DOWN=double spacing
    1,  # 2-4: UP=7-bit data, DOWN=8-bit data
    1,  # 2-5: UP=Even parity, DOWN=Odd parity
        #
    1,  # 2-6:  DOWN    DOWN    DOWN    DOWN    UP      UP      UP      UP
    1,  # 2-7:  DOWN    DOWN    UP      UP      DOWN    DOWN    UP      UP
    1,  # 2-8:  DOWN    UP      DOWN    UP      DOWN    UP      DOWN    UP
        # BAUD  9600    4800    2400    1200    600     300     150     110
]

DIP_SWITCHES = sum(bit << i for i, bit in enumerate(switches))

def setup_scope(time_scale=1.5e-4, delay=4.5e-4, trigger_channel=4, trigger_slope="NEGative"):
    """
    Quick scope setup for notebook use
    """
    scope = KeysightInfiniiVisionMSO(ADDRESS)

    scope.setup_digital_channels(
        channels=range(6),
        threshold=2.5,
        time_scale=time_scale,
    )

    for i, label in enumerate(DEFAULT_LABELS):
        scope.set_digital_label(i, label)

    scope.setup_digital_trigger(channel=trigger_channel, slope=trigger_slope)
    scope.set_trigger_delay(delay)

    return scope

In [ ]:
"""
Brother Serial Interface - Automated Protocol Test Suite
"""

from datetime import datetime
from TestDefinitions import (
	TESTS_CONTROL,
	TESTS_PRINTABLE,
	TESTS_MOVEMENT,
	TESTS_SPACING,
	TESTS_MODES,
	TESTS_MARGINS_H,
	TESTS_TABS,
	TESTS_SPECIAL,
	TESTS_PAPERFEED,
)
# ---------------------------------------------------------------------------
# Configuration — change these to control what runs
# ---------------------------------------------------------------------------
TESTS = list()
TESTS += TESTS_CONTROL
# TESTS += TESTS_PRINTABLE
# TESTS += TESTS_SPECIAL
TESTS += TESTS_MOVEMENT
# TESTS += TESTS_SPACING
# TESTS += TESTS_MODES
TESTS += TESTS_MARGINS_H
# TESTS += TESTS_TABS
# TESTS += TESTS_PAPERFEED
NO_PROMPT = True  # Set True to run without pausing between tests

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
log_lines = []


def log(line=""):
    log_lines.append(line)
    print(line)


# ---------------------------------------------------------------------------
# Run a Single Test
# ---------------------------------------------------------------------------
def run_test(test, scope, con):
    """Execute all steps in a test, capturing where indicated."""
    log(test.fmt_header())
    if TYPEWRITER[0] not in test.compatible_series:
        log("Test not compatible with Series '= None expected' - SKIPPED")
        log()
        return

    for step in test.steps:
        if step.capture:
            scope.scope.clear_errors()
            scope.arm_trigger()
            time.sleep(0.2)
            con.write(bytes(step.bytes))
            time.sleep(step.wait)

            si = None
            so = None
            try:
                scope.scope.clear_errors()
                time_data, packed, channel_data = scope.read_current_pod_data(
                    pod=1, mode='RAW'
                )
                raw_decoder = RawDecoder(channel_data, time_data)
                decoded_si = BrotherSerialDecoder(
                    raw_decoder.decode(data_ch=0)
                ).decode()
                decoded_so = BrotherSerialDecoder(
                    raw_decoder.decode(data_ch=1)
                ).decode()
                si = [b.raw_value for b in decoded_si.bytes]
                so = [b.raw_value for b in decoded_so.bytes]
            except Exception:
                pass

            log(step.fmt_result(si, so))
        else:
            con.write(bytes(step.bytes))
            time.sleep(step.wait)

    log()


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
log(f"// Brother Serial Interface Protocol Test")
log(f"// Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log(f"// Interface: {INTERFACE}")
log(f"// Typewriter: {TYPEWRITER}")
log(f"// Keyboard: {KEYBOARD}")
log(f"// Interface PITCH Selection: {IFPITCH} (Default 10)")
log(f"// DIP Switches: 0b{DIP_SWITCHES:013b} (0x{DIP_SWITCHES:04X})")
log(f"//")
log(f"//   1-1: {'DOWN' if switches[0] else 'UP':4s}  UP=RS-232C, DOWN=CDCC interface")
log(f"//   1-2: {'DOWN' if switches[1] else 'UP':4s}  UP=Terminal mode, DOWN=Printer mode")
log(f"//   1-3: {'DOWN' if switches[2] else 'UP':4s}  USA users can ignore")
log(f"//   1-4: {'DOWN' if switches[3] else 'UP':4s}  UP=ASCII Wheel, DOWN=Non-ASCII Wheel")
log(f"//   1-5: {'DOWN' if switches[4] else 'UP':4s}  UP=12-inch paper, DOWN=11-inch paper")
log(f"//   1-6: {'DOWN' if switches[5] else 'UP':4s}  UP=Auto skip perforation, DOWN=Non auto skip")
log(f"//   2-1: {'DOWN' if switches[6] else 'UP':4s}  UP=Local echo (half-duplex), DOWN=No echo (full-duplex)")
log(f"//   2-2: {'DOWN' if switches[7] else 'UP':4s}  UP=DC-1/DC-3 disabled, DOWN=enabled")
log(f"//   2-3: {'DOWN' if switches[8] else 'UP':4s}  UP=Auto line feed off, DOWN=Double spacing")
log(f"//   2-4: {'DOWN' if switches[9] else 'UP':4s}  UP=7-bit data, DOWN=8-bit data")
log(f"//   2-5: {'DOWN' if switches[10] else 'UP':4s}  UP=Even parity, DOWN=Odd parity")
log(f"//   2-6..2-8: {switches[11]}{switches[12]}{switches[13]}  Baud: " +
    f"{'9600' if switches[11:14]==[1,1,1] else '4800' if switches[11:14]==[1,1,0] else '2400' if switches[11:14]==[1,0,1] else '1200' if switches[11:14]==[1,0,0] else '600' if switches[11:14]==[0,1,1] else '300' if switches[11:14]==[0,1,0] else '150' if switches[11:14]==[0,0,1] else '110'}")
log(f"//")
log()

with setup_scope(
    time_scale=5e-3, delay=20e-3, trigger_slope="NEGative"
) as scope:
    with serial.Serial(
        port='/dev/ttyUSB0',
        baudrate=9600,
        bytesize=serial.EIGHTBITS,
        stopbits=serial.STOPBITS_ONE,
        parity=serial.PARITY_NONE,
        rtscts=True,
        timeout=1,
    ) as con:

        for i, test in enumerate(TESTS):
            if not NO_PROMPT:
                choice = input(f"[{i}] {test.comment} (Enter/q): ").strip()
                if choice == 'q':
                    log("// TESTRUN ABORTED")
                    break

            run_test(test, scope, con)

# ---------------------------------------------------------------------------
# Save Log
# ---------------------------------------------------------------------------
filename = (
    f"test_{TYPEWRITER}_{INTERFACE}_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
)
with open(filename, 'w') as f:
    f.write("\n".join(log_lines))

log(f"// Log saved to: {filename}")

// Brother Serial Interface Protocol Test
// Date: 2026-02-13 10:59:46
// Interface: IF60
// Typewriter: AX20
// Keyboard: 1
// Interface PITCH Selection: 15 (Default 10)
// DIP Switches: 0b11111111111000 (0x3FF8)
//
//   1-1: UP    UP=RS-232C, DOWN=CDCC interface
//   1-2: UP    UP=Terminal mode, DOWN=Printer mode
//   1-3: UP    USA users can ignore
//   1-4: DOWN  UP=ASCII Wheel, DOWN=Non-ASCII Wheel
//   1-5: DOWN  UP=12-inch paper, DOWN=11-inch paper
//   1-6: DOWN  UP=Auto skip perforation, DOWN=Non auto skip
//   2-1: DOWN  UP=Local echo (half-duplex), DOWN=No echo (full-duplex)
//   2-2: DOWN  UP=DC-1/DC-3 disabled, DOWN=enabled
//   2-3: DOWN  UP=Auto line feed off, DOWN=Double spacing
//   2-4: DOWN  UP=7-bit data, DOWN=8-bit data
//   2-5: DOWN  UP=Even parity, DOWN=Odd parity
//   2-6..2-8: 111  Baud: 9600
//

Connected to: AGILENT TECHNOLOGIES,MSO7014A,MY49110266,06.20.0000
Digital channels [0, 1, 2, 3, 4, 5] enabled, threshold=2.5V
Edge trigger on D4, slope=NEGative
// 0x